# Beginner MCP Client Workflow: Discover and Read Homer

## Table of contents (ToC) <a class="anchor" id="TOC"></a>
* <a href="#introduction">1 - Introduction and learning goals</a>
* <a href="#mcp-concepts">2 - MCP concepts used in this notebook</a>
* <a href="#setup">3 - Install dependencies and load the local server</a>
* <a href="#helpers">4 - Understand MCP results and helper functions</a>
* <a href="#connect">5 - Open an MCP session and list tools</a>
* <a href="#schemas">6 - Inspect tool descriptions and input schemas</a>
* <a href="#author-discovery">7 - Discover the correct Homer textgroup</a>
* <a href="#resource-discovery">8 - Discover Homer's works and editions</a>
* <a href="#select-iliad">9 - Select the Iliad and a Greek CTS edition</a>
* <a href="#references">10 - Inspect valid passage references</a>
* <a href="#raw-plaintext">11 - Compare raw XML with plaintext</a>
* <a href="#fetch-lines">12 - Fetch the first five lines through MCP</a>
* <a href="#analysis">13 - Analyze the returned Greek with Python</a>
* <a href="#transport">14 - In-process transport versus an external MCP client</a>
* <a href="#troubleshooting">15 - Troubleshooting and safe habits</a>
* <a href="#next-steps">16 - Continue learning</a>
* <a href="#sources">17 - Sources</a>
* <a href="#required-libraries">18 - Required libraries</a>
* <a href="#notebook-version">19 - Notebook version</a>

## 1 - Introduction and learning goals <a class="anchor" id="introduction"></a>
##### [Back to ToC](#TOC)

This is the first notebook in the examples directory that makes **actual MCP tool calls**. Notebooks `01_` and `02_` call Perseus CTS and Scaife directly with HTTP. This notebook instead connects a FastMCP `Client` to the local `perseus` MCP server and asks the server to perform discovery and passage retrieval.

The research workflow is deliberately small and concrete:

1. load the local MCP server defined in [`src/perseus_mcp/server.py`](../src/perseus_mcp/server.py);
2. open a client session and inspect the registered tools;
3. search author names so that `Homer` is distinguished from `Homeric Hymns`;
4. retrieve Homer's works and choose the *Iliad*;
5. select a Greek edition advertised by the current CTS inventory;
6. request valid references instead of guessing citations;
7. fetch the opening five lines as readable Greek text;
8. run a small, transparent Python analysis on the result.

No OpenRouter key or other API key is needed. The local MCP server sends ordinary HTTP requests to the public Perseus service.

## 2 - MCP concepts used in this notebook <a class="anchor" id="mcp-concepts"></a>
##### [Back to ToC](#TOC)

**MCP** stands for **Model Context Protocol**. It defines a standard way for a client to discover and call capabilities exposed by a server. Although MCP is often used by an LLM application, the client in this notebook is ordinary Python code.

| Concept | Meaning in this notebook |
|---|---|
| MCP server | The `FastMCP("perseus")` object created in `perseus_mcp.server` |
| Tool | A named operation registered with `@mcp.tool`, such as `get_author_resources` |
| Client | `fastmcp.Client`, used to list and call the registered tools |
| Tool schema | JSON metadata describing a tool's accepted arguments |
| Session | The connection opened by `async with Client(mcp) as client` |
| Transport | How client and server exchange MCP messages |
| Content block | A typed result item returned by an MCP tool call |

The data flow is:

```text
notebook code
    → FastMCP Client
        → MCP tool name + validated arguments
            → local tool in perseus_mcp.server
                → Perseus CTS HTTP service
            ← tool response
        ← MCP content blocks
    ← readable Python value
```

This notebook uses an **in-process transport**: the client and server object live in the same Python process. It still calls tools through FastMCP's MCP interface; it does not call `server.get_passage_plaintext(...)` directly.

## 3 - Install dependencies and load the local server <a class="anchor" id="setup"></a>
##### [Back to ToC](#TOC)

The recommended project installation is `pip install -e .` or `uv sync` from the repository root. The first cell below also installs the runtime libraries into the active Jupyter kernel so the notebook can be opened in a fresh environment.

The setup code then:

- searches the current directory and its parents for `src/perseus_mcp/server.py`;
- adds the repository's `src` directory to `sys.path` so `from perseus_mcp import server` works;
- points notebook cache files at the repository-level `.cache/perseus-mcp` directory;
- imports and reloads the local `perseus_mcp.server` module;
- stores its registered FastMCP server object in `mcp`.

Importing `perseus_mcp.server` does **not** start a command-line server. The `mcp.run()` call is only invoked through the package's command-line entry point.

In [1]:
%pip install --quiet "fastmcp>=2.12.0" "httpx>=0.27.0" "python-dotenv>=1.0.0"

Note: you may need to restart the kernel to use updated packages.


In [2]:
from collections import Counter
from pathlib import Path
import importlib
import json
import os
import re
import sys

START = Path.cwd().resolve()
for candidate in [START, *START.parents]:
    if (candidate / "src" / "perseus_mcp" / "server.py").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError(
        f"Could not find src/perseus_mcp/server.py from {START}. Open this notebook inside the Perseus-mcp repository."
    )

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Keep notebook and command-line runs pointed at the same metadata cache.
os.environ.setdefault(
    "PERSEUS_MCP_CACHE_DIR",
    str(REPO_ROOT / ".cache" / "perseus-mcp"),
)

from fastmcp import Client
from perseus_mcp import server

# Reload local edits when the notebook is rerun in an existing kernel.
server = importlib.reload(server)
mcp = server.mcp

print(f"Repository root: {REPO_ROOT}")
print(f"Cache directory: {os.environ['PERSEUS_MCP_CACHE_DIR']}")
print(f"Loaded MCP server object: {mcp.name}")

Repository root: D:\Onedrive\GitHub\Perseus-mcp
Cache directory: D:\Onedrive\GitHub\Perseus-mcp\.cache\perseus-mcp
Loaded MCP server object: perseus


## 4 - Understand MCP results and helper functions <a class="anchor" id="helpers"></a>
##### [Back to ToC](#TOC)

`client.call_tool(...)` does not normally return a bare string or dictionary. It returns a FastMCP call-result object containing one or more **content blocks**. The tools in this repository return text payloads, but that text may represent:

- readable passage text;
- raw XML;
- serialized JSON.

The helpers below keep those cases explicit:

- `tool_text` extracts all text content blocks;
- `tool_json` parses a JSON text payload into Python objects;
- `call_text` and `call_json` perform a named MCP call using an existing client session;
- `print_json` displays a Python value as readable Unicode JSON.

Parsing is done in the notebook because MCP transports content; the caller still decides how to use that content.

In [3]:
def tool_text(result):
    """Extract all text content blocks from a FastMCP call result."""
    return "\n".join(
        block.text
        for block in result.content
        if getattr(block, "text", None) is not None
    )


def tool_json(result):
    """Parse a tool's text payload as JSON."""
    return json.loads(tool_text(result))


async def call_text(client, tool_name, arguments=None):
    result = await client.call_tool(tool_name, arguments or {})
    return tool_text(result)


async def call_json(client, tool_name, arguments=None):
    result = await client.call_tool(tool_name, arguments or {})
    return tool_json(result)


def print_json(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))

## 5 - Open an MCP session and list tools <a class="anchor" id="connect"></a>
##### [Back to ToC](#TOC)

`Client(mcp)` tells FastMCP to connect to the imported server object with an in-process transport. The `async with` block opens the session and closes it cleanly afterward.

Jupyter supports top-level `await`, so an `async with` block can be used directly in a cell. In an ordinary Python script you would usually place the same code inside an `async def main()` function and run that function with `asyncio.run(main())`.

`list_tools()` is a useful first call because it proves the connection works without making a request to Perseus. It returns the catalog that an external MCP-capable application would also see.

In [4]:
async with Client(mcp) as client:
    tools = await client.list_tools()
    cache_status = await call_json(client, "get_cache_status")

print(f"Registered tools: {len(tools)}")
for index, tool in enumerate(tools, start=1):
    summary = (tool.description or "No description provided.").splitlines()[0]
    print(f"{index:>2}. {tool.name}: {summary}")

print("\nLocal metadata cache:")
print_json(cache_status)

Registered tools: 23
 1. get_passage: Get the text of a specific passage using a CTS URN.
 2. get_passage_plus: Get passage text plus surrounding metadata/context for a CTS URN.
 3. get_passage_plaintext: Get a passage as plain readable text instead of raw CTS XML.
 4. get_valid_references: Get valid citations/references for a work, useful for navigation.
 5. get_valid_references_json: Get valid citation references as paged JSON instead of raw CTS XML.
 6. count_valid_references: Count valid citation references without returning the full reference list.
 7. get_capabilities: Get the list of available texts and editions from Perseus CTS.
 8. get_cache_status: Get local metadata cache status.
 9. refresh_metadata_cache: Refresh cached CTS capabilities metadata from Perseus.
10. clear_metadata_cache: Clear local metadata cache files and in-memory cache entries.
11. list_text_groups: List authors/textgroups and their works from CTS capabilities.
12. get_author_resources: List CTS works/edi

## 6 - Inspect tool descriptions and input schemas <a class="anchor" id="schemas"></a>
##### [Back to ToC](#TOC)

Every MCP tool advertises a name, a natural-language description, and a JSON input schema. The schema is important because it tells clients—and tool-using models—which argument names and types are valid before a call is made.

The selected tools below illustrate the three stages of this workflow:

- discover an author with a partial name;
- inspect the resources belonging to an exact author/textgroup;
- retrieve a passage once a valid URN has been selected.

In [5]:
tool_by_name = {tool.name: tool for tool in tools}

for name in [
    "find_author_names",
    "get_author_resources",
    "get_valid_references_json",
    "get_passage_plaintext",
]:
    tool = tool_by_name[name]
    print(f"\n{name}")
    print(tool.description)
    print("Input schema:")
    print(json.dumps(tool.inputSchema, ensure_ascii=False, indent=2))


find_author_names
Find author/textgroup names by partial name match.

This matches only exact CTS author/textgroup name fields, not work titles.
Examples:
- query: "Hom"
- query: "Plut"
Input schema:
{
  "additionalProperties": false,
  "properties": {
    "query": {
      "type": "string"
    },
    "language": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null
    },
    "limit": {
      "default": 100,
      "type": "integer"
    }
  },
  "required": [
    "query"
  ],
  "type": "object"
}

get_author_resources
List CTS works/editions/translations for an author name or textgroup URN.

Examples:
- author: "Homer"
- author: "tlg0012"
- author: "urn:cts:greekLit:tlg0012"
Input schema:
{
  "additionalProperties": false,
  "properties": {
    "author": {
      "type": "string"
    },
    "language": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "nu

## 7 - Discover the correct Homer textgroup <a class="anchor" id="author-discovery"></a>
##### [Back to ToC](#TOC)

A beginner might immediately call `get_author_resources` with `"Hom"` or `"Homer"`. A safer workflow begins with `find_author_names`, especially when the query may be partial or ambiguous.

The query `Hom` can match both `Homer` and `Homeric Hymns`. The response includes stable CTS textgroup URNs, which are better identifiers than names alone. The code selects the row whose advertised name is exactly `Homer`.

In [6]:
async with Client(mcp) as client:
    author_matches = await call_json(
        client,
        "find_author_names",
        {"query": "Hom", "language": "greek", "limit": 10},
    )

compact_author_matches = [
    {
        "urn": author["urn"],
        "names": author["names"],
        "matched_names": author["matched_names"],
        "works_count": author["works_count"],
    }
    for author in author_matches["authors"]
]
print_json(compact_author_matches)

homer_match = next(
    (author for author in author_matches["authors"] if "Homer" in author["names"]),
    None,
)
if homer_match is None:
    raise RuntimeError("The current CTS inventory did not return an exact Homer match.")

HOMER_TEXTGROUP = homer_match["urn"]
print(f"\nSelected textgroup: {HOMER_TEXTGROUP}")

[
  {
    "urn": "urn:cts:greekLit:tlg0013",
    "names": [
      "Homeric Hymns"
    ],
    "matched_names": [
      "Homeric Hymns"
    ],
    "works_count": 33
  },
  {
    "urn": "urn:cts:greekLit:tlg0012",
    "names": [
      "Homer"
    ],
    "matched_names": [
      "Homer"
    ],
    "works_count": 2
  }
]

Selected textgroup: urn:cts:greekLit:tlg0012


## 8 - Discover Homer's works and editions <a class="anchor" id="resource-discovery"></a>
##### [Back to ToC](#TOC)

`get_author_resources` accepts either a human-readable author query or a CTS textgroup URN. Passing the exact URN selected above avoids mixing Homer with similarly named textgroups.

The `language="greek"` argument filters the advertised **works** to Greek-language works. Each work can still contain several resources, such as editions and translations. The full response is nested, so the notebook first prints a compact summary before selecting anything.

In [7]:
async with Client(mcp) as client:
    homer_resources = await call_json(
        client,
        "get_author_resources",
        {"author": HOMER_TEXTGROUP, "language": "greek"},
    )

if not homer_resources["authors"]:
    raise RuntimeError("No Homer resources were returned by the current CTS inventory.")

homer_author = homer_resources["authors"][0]
homer_work_summary = [
    {
        "work_urn": work["urn"],
        "titles": work["titles"],
        "language": work["language"],
        "editions": [edition.get("urn") for edition in work["editions"]],
        "translations": [translation.get("urn") for translation in work["translations"]],
    }
    for work in homer_author["works"]
]

print(f"Author names: {homer_author['names']}")
print_json(homer_work_summary)

Author names: ['Homer']
[
  {
    "work_urn": "urn:cts:greekLit:tlg0012.tlg001",
    "titles": [
      "Iliad"
    ],
    "language": "grc",
    "editions": [
      "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1"
    ],
    "translations": [
      "urn:cts:greekLit:tlg0012.tlg001.perseus-eng1",
      "urn:cts:greekLit:tlg0012.tlg001.perseus-eng2"
    ]
  },
  {
    "work_urn": "urn:cts:greekLit:tlg0012.tlg002",
    "titles": [
      "Odyssey"
    ],
    "language": "grc",
    "editions": [
      "urn:cts:greekLit:tlg0012.tlg002.perseus-grc1"
    ],
    "translations": [
      "urn:cts:greekLit:tlg0012.tlg002.perseus-eng1",
      "urn:cts:greekLit:tlg0012.tlg002.perseus-eng2"
    ]
  }
]


## 9 - Select the Iliad and a Greek CTS edition <a class="anchor" id="select-iliad"></a>
##### [Back to ToC](#TOC)

The next step converts a human intention—"read the Greek *Iliad*"—into an edition-specific CTS URN.

The code does not hard-code an edition suffix such as `perseus-grc1`. It:

1. finds a work whose title is exactly `Iliad`;
2. examines the editions advertised below that work;
3. selects a Greek edition by its language field or Greek-looking URN;
4. confirms the work through the separate `get_work_resources` tool.

Discovery before construction is important because upstream inventories can change and Scaife search editions may use different URNs from Perseus CTS.

In [8]:
iliad_work = next(
    (work for work in homer_author["works"] if "Iliad" in work["titles"]),
    None,
)
if iliad_work is None:
    raise RuntimeError("The current Homer resources contain no work titled Iliad.")


def is_greek_edition(resource):
    urn = resource.get("urn") or ""
    return resource.get("language") == "grc" or "-grc" in urn or ".perseus-grc" in urn


greek_iliad_editions = [
    edition for edition in iliad_work["editions"] if is_greek_edition(edition)
]
if not greek_iliad_editions:
    raise RuntimeError("The current CTS inventory advertises no Greek Iliad edition.")

ILIAD_WORK = iliad_work["urn"]
ILIAD_EDITION = greek_iliad_editions[0]["urn"]

async with Client(mcp) as client:
    iliad_confirmation = await call_json(
        client,
        "get_work_resources",
        {"urn_or_title": ILIAD_WORK},
    )

print_json(
    {
        "work_urn": ILIAD_WORK,
        "titles": iliad_work["titles"],
        "available_greek_editions": greek_iliad_editions,
        "selected_edition": ILIAD_EDITION,
        "work_confirmation_matches": iliad_confirmation["match_count"],
    }
)

{
  "work_urn": "urn:cts:greekLit:tlg0012.tlg001",
  "titles": [
    "Iliad"
  ],
  "available_greek_editions": [
    {
      "type": "edition",
      "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
      "label": "Iliad",
      "description": "Perseus:bib:oclc,29448041, Homer. Homeri Opera in five volumes. Oxford, Oxford University Press. 1920."
    }
  ],
  "selected_edition": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
  "work_confirmation_matches": 1
}


## 10 - Inspect valid passage references <a class="anchor" id="references"></a>
##### [Back to ToC](#TOC)

A valid edition URN still does not tell us which citations exist. `count_valid_references` and `get_valid_references_json` answer that question without returning the complete, potentially large XML reference list.

- `count_valid_references` is useful when only the size is needed;
- `get_valid_references_json` returns a page controlled by `limit` and `offset`;
- `has_next` tells the caller whether another page exists.

The first five advertised references will become the input for the passage calls below. This avoids guessing that `1.1` through `1.5` exist and avoids depending on a range response from the live CTS service.

In [9]:
async with Client(mcp) as client:
    reference_count = await call_json(
        client,
        "count_valid_references",
        {"urn": ILIAD_EDITION},
    )
    first_reference_page = await call_json(
        client,
        "get_valid_references_json",
        {"urn": ILIAD_EDITION, "limit": 5, "offset": 0},
    )

print("Reference count:")
print_json(reference_count)
print("\nFirst reference page:")
print_json(first_reference_page)

opening_urns = first_reference_page["references"]
if len(opening_urns) < 5:
    raise RuntimeError("Fewer than five Iliad references were returned.")

Reference count:
{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
  "level": null,
  "total_count": 14956
}

First reference page:
{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
  "level": null,
  "total_count": 14956,
  "offset": 0,
  "limit": 5,
  "returned_count": 5,
  "has_next": true,
  "references": [
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.2",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.3",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.4",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.5"
  ]
}


## 11 - Compare raw XML with plaintext <a class="anchor" id="raw-plaintext"></a>
##### [Back to ToC](#TOC)

Two tools can retrieve the same passage in different forms:

- `get_passage` returns the raw CTS XML supplied by Perseus;
- `get_passage_plaintext` asks the local server to extract readable text from that XML.

This illustrates an important MCP design choice: tools can expose a low-level source response when fidelity matters and a higher-level convenience representation when ordinary text processing is the goal.

The cell also inspects the FastMCP result object before using the helper, so the relationship between a call result, content blocks, and the extracted payload is visible.

In [10]:
first_urn = opening_urns[0]

async with Client(mcp) as client:
    raw_result = await client.call_tool("get_passage", {"urn": first_urn})
    plaintext_result = await client.call_tool(
        "get_passage_plaintext",
        {"urn": first_urn},
    )

raw_xml = tool_text(raw_result)
first_line_text = tool_text(plaintext_result)

print(f"Call result type: {type(raw_result).__name__}")
print(f"Content blocks: {len(raw_result.content)}")
print(f"Block types: {[type(block).__name__ for block in raw_result.content]}")
print("\nRaw XML prefix:")
print(raw_xml[:900])
print("\nPlaintext:")
print(first_line_text)

Call result type: CallToolResult
Content blocks: 1
Block types: ['TextContent']

Raw XML prefix:
<?xml version="1.0" encoding="UTF-8"?>
<cts:GetPassage xmlns:m="http://mulberrytech.com/xslt/util"
                xmlns:cts="http://chs.harvard.edu/xmlns/cts3"
                xmlns:tei="http://www.tei-c.org/ns/1.0">
   <cts:request>
      <cts:requestName>GetPassage</cts:requestName>
      <cts:requestUrn>urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1</cts:requestUrn>
      <cts:psg>1.1</cts:psg>
      <cts:workUrn>urn:cts:greekLit:tlg0012.tlg001.perseus-grc1</cts:workUrn>
      <cts:groupname>Homer</cts:groupname>
      <cts:title>Iliad</cts:title>
      <cts:label>Iliad</cts:label>
      <cts:versionInfo>Perseus 4.0</cts:versionInfo>
   </cts:request>
   <cts:reply>
      <tei:TEI>
         <tei:text xml:lang="grc">
            <tei:body>
               <tei:div type="line">
                  <milestone ed="P" unit="para"/>μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος</tei:div>
            </t

P

## 12 - Fetch the first five lines through MCP <a class="anchor" id="fetch-lines"></a>
##### [Back to ToC](#TOC)

A single client session can make several tool calls. The loop below calls `get_passage_plaintext` once for each of the first five valid URNs.

This approach is intentionally explicit:

- every requested line was obtained from the reference-discovery tool;
- every retrieval goes through `client.call_tool`;
- each result remains paired with its source URN;
- the server handles XML retrieval and plaintext conversion.

For a larger corpus workflow, consider concurrency, rate limits, caching, and saving intermediate results rather than issuing an unbounded loop of live requests.

In [12]:
passage_lines = []

async with Client(mcp) as client:
    for urn in opening_urns:
        text = await call_text(
            client,
            "get_passage_plaintext",
            {"urn": urn},
        )
        passage_lines.append({"urn": urn, "text": text})

for line in passage_lines:
    citation = line["urn"].rpartition(":")[2]
    print(f"{citation:>4}  {line['text']}")

greek_text = "\n".join(line["text"] for line in passage_lines)

 1.1  μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος
 1.2  οὐλομένην, ἣ μυρί᾽ Ἀχαιοῖς ἄλγε᾽ ἔθηκε,
 1.3  πολλὰς δ᾽ ἰφθίμους ψυχὰς Ἄϊδι προΐαψεν
 1.4  ἡρώων, αὐτοὺς δὲ ἑλώρια τεῦχε κύνεσσιν
 1.5  οἰωνοῖσί τε πᾶσι, Διὸς δ᾽ ἐτελείετο βουλή,


## 13 - Analyze the returned Greek with Python <a class="anchor" id="analysis"></a>
##### [Back to ToC](#TOC)

Once the MCP payload has been converted to an ordinary Python string, no special MCP code is needed. The simple tokenizer below keeps characters in the Greek and Greek Extended Unicode blocks, lowercases the text, and counts visible forms.

This is intentionally a small demonstration, not a complete linguistic tokenizer. It does not normalize accents, lemmatize words, resolve elision, or account for punctuation and editorial markup. For lexical research, use Scaife lemma search or a dedicated Greek NLP workflow.

In [13]:
tokens = re.findall(r"[Ͱ-Ͽἀ-῿]+", greek_text.lower())
token_counts = Counter(tokens)

print(f"Passages: {len(passage_lines)}")
print(f"Greek tokens: {len(tokens)}")
print(f"Distinct visible forms: {len(token_counts)}")
print("\nMost frequent visible forms:")
for token, count in token_counts.most_common(10):
    print(f"{token}\t{count}")

Passages: 5
Greek tokens: 30
Distinct visible forms: 29

Most frequent visible forms:
δ᾽	2
μῆνιν	1
ἄειδε	1
θεὰ	1
πηληϊάδεω	1
ἀχιλῆος	1
οὐλομένην	1
ἣ	1
μυρί᾽	1
ἀχαιοῖς	1


## 14 - In-process transport versus an external MCP client <a class="anchor" id="transport"></a>
##### [Back to ToC](#TOC)

This notebook imports `perseus_mcp.server.mcp` and passes that object directly to `Client`. That is convenient for examples, tests, and local development because no separate background process is needed.

| In this notebook | External desktop or coding client |
|---|---|
| Client and server object share one Python process | The client launches the `perseus-mcp` command as a separate process |
| Transport is in-process | Transport is normally standard input/output (stdio) |
| Repository `src` directory is imported with `sys.path` | Client configuration supplies the installed command or repository launch command |
| Excellent for inspecting Python values | Appropriate for Claude Desktop, Cursor, Windsurf, Codex, or other MCP hosts |
| Tool names, schemas, validation, and implementations are the same | Tool names, schemas, validation, and implementations are the same |

A typical external launch command is:

```bash
uv --directory /full/path/to/Perseus-mcp run perseus-mcp
```

The external client does not need to know the Perseus HTTP details. It discovers the MCP tools and calls them by name with schema-valid arguments.

## 15 - Troubleshooting and safe habits <a class="anchor" id="troubleshooting"></a>
##### [Back to ToC](#TOC)

| Symptom | Likely cause and response |
|---|---|
| `ModuleNotFoundError: fastmcp` | Run the installation cell in the active kernel, then rerun setup |
| `Could not find src/perseus_mcp/server.py` | Open the notebook from within the cloned repository or adjust the root-discovery code |
| A new local tool is missing | Rerun the setup cell so `importlib.reload(server)` refreshes the registry; restart the kernel if necessary |
| No author or edition is returned | Rerun discovery; the live CTS inventory may have changed, or the language/query may be too restrictive |
| A live request fails | Confirm internet access and retry later; Perseus and Scaife are upstream services |
| Metadata appears stale | Call `refresh_metadata_cache`, or inspect `get_cache_status` before clearing anything |
| An `OSError` occurs while reading cached metadata | Use `clear_metadata_cache` in a fresh session, or point `PERSEUS_MCP_CACHE_DIR` at a local writable directory that is not holding a corrupt/cloud-placeholder file |
| A hard-coded edition stops working | Replace it with a URN selected from `get_author_resources` or `get_work_resources` |
| Async code is confusing | In Jupyter, use top-level `await`/`async with`; in a script, use an async `main()` function |

Safe habits for research notebooks:

- discover resources before constructing edition-specific URNs;
- keep the URN beside every extracted passage;
- distinguish raw XML, serialized JSON, and plaintext tool payloads;
- use paged/count tools instead of moving very large metadata responses unnecessarily;
- record the execution date when upstream data matters;
- clear or review outputs before committing notebooks that use credentials or private data (this notebook uses neither).

## 16 - Continue learning <a class="anchor" id="next-steps"></a>
##### [Back to ToC](#TOC)

Useful next experiments:

- change the `find_author_names` query and inspect another author's works;
- select the *Odyssey* from `homer_author["works"]` and repeat the reference/passage workflow;
- change the `offset` in `get_valid_references_json` to page through citations;
- call `get_prev_next_urn` for one of the discovered passage URNs;
- compare `get_passage`, `get_passage_plus`, and `get_passage_plaintext`;
- inspect all tools and schemas in [`05_mcp_all_tools.ipynb`](05_mcp_all_tools.ipynb).

Continue with:

- [`04_mcp_greek_search_and_navigation.ipynb`](04_mcp_greek_search_and_navigation.ipynb) for Unicode/Beta Code search and navigation through MCP;
- [`07_mcp_advanced_search_options.ipynb`](07_mcp_advanced_search_options.ipynb) for form, lemma, operator, and author-scoped search;
- [`08_mcp_cache_and_search_tools.ipynb`](08_mcp_cache_and_search_tools.ipynb) for caching, reference paging, edition-scoped search, highlights, and Scaife-native retrieval;
- [`06_openrouter_llm_mcp_interaction.ipynb`](06_openrouter_llm_mcp_interaction.ipynb) only after the direct Python MCP workflow here is comfortable.

## 17 - Sources <a class="anchor" id="sources"></a>
##### [Back to ToC](#TOC)

This notebook uses:

- the local Perseus MCP implementation in [`src/perseus_mcp/server.py`](../src/perseus_mcp/server.py);
- the project setup and client guidance in the [README](../README.md);
- [FastMCP](https://github.com/jlowin/fastmcp) for the client, server, tool registry, schemas, validation, and in-process transport;
- the [Perseus Digital Library](https://www.perseus.tufts.edu/) CTS service for the live inventory, references, and Greek text.

The server also contains Scaife search and retrieval tools, but this particular workflow only calls CTS-backed discovery, cache, reference, and passage tools. Upstream inventories and responses may change, which is why the notebook discovers resources at runtime.

## 18 - Required libraries <a class="anchor" id="required-libraries"></a>
##### [Back to ToC](#TOC)

The repository requires **Python 3.11 or newer**. Its declared runtime dependencies are:

- `fastmcp>=2.12.0` — MCP client/server functionality;
- `httpx>=0.27.0` — upstream HTTP requests made inside the server;
- `python-dotenv>=1.0.0` — optional environment-file loading used by the project.

Jupyter/IPython is also needed to run this notebook. `collections`, `json`, `pathlib`, `re`, `os`, and `sys` are Python standard-library modules.

Recommended installation from the repository root:

```bash
pip install -e .
```

or:

```bash
uv sync
```

The notebook installation cell is a convenience for a fresh kernel; an editable project installation is preferable for repeated development.

## 19 - Notebook version <a class="anchor" id="notebook-version"></a>
##### [Back to ToC](#TOC)

<div style="float: left;">
  <table>
    <tr>
      <td><strong>Author</strong></td>
      <td>Tony Jurg</td>
    </tr>
    <tr>
      <td><strong>Version</strong></td>
      <td>1.2</td>
    </tr>
    <tr>
      <td><strong>Date</strong></td>
      <td>June 18, 2026</td>
    </tr>
  </table>
</div>